In [1]:
# =============================================================================
# File paths
# =============================================================================

import ast
import pandas as pd

In [2]:
# =============================================================================
# File paths
# =============================================================================

DATA_FILE = "LIS_data_cleaned_with_OND.csv"
CLASSIFICATION_FILE = "deployment_llama3_70b_LIS_classification.csv"

OUTPUT_FILE = "LIS_data_cleaned_with_OND_Class.csv"

In [3]:
# =============================================================================
# Column names
# =============================================================================

FUNDING_COLUMN = "Funding_Details_OND"
CLASSIFIED_ORG_COLUMN = "Representative_Organization_Name"
CLASS_COLUMN = "Class"

NEW_COLUMN = "Funding_Details_OND_Class"

In [4]:
# =============================================================================
# Load the datasets
# =============================================================================

df = pd.read_csv(
    DATA_FILE,
    encoding="utf-8-sig"
)

classification_df = pd.read_csv(
    CLASSIFICATION_FILE,
    encoding="utf-8-sig"
)

In [5]:
# =============================================================================
# Create the organization-to-class mapping
# =============================================================================

org_class_mapping = dict(
    zip(
        classification_df[CLASSIFIED_ORG_COLUMN],
        classification_df[CLASS_COLUMN]
    )
)

In [6]:
# =============================================================================
# Parse the organization lists safely
# =============================================================================

def parse_organization_list(value):
    """
    Convert a string representation of a Python list into an actual list.
    Empty or missing values are returned as empty lists.
    """

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        parsed_value = ast.literal_eval(value)

        if isinstance(parsed_value, list):
            return parsed_value

        return []

    except (ValueError, SyntaxError):
        return []


In [7]:
# =============================================================================
# Identify unmatched organizations before manual assignment
# =============================================================================

all_funding_organizations = set()

for value in df[FUNDING_COLUMN]:
    all_funding_organizations.update(
        parse_organization_list(value)
    )

classified_organizations = set(
    classification_df[
        CLASSIFIED_ORG_COLUMN
    ].dropna()
)

unmatched_organizations = sorted(
    all_funding_organizations
    - classified_organizations
)

print(
    f"Number of unique organizations in Funding_Details_OND: "
    f"{len(all_funding_organizations):,}"
)

print(
    f"Number of unmatched organizations: "
    f"{len(unmatched_organizations):,}"
)

print("\nUnmatched organizations:")

for organization in unmatched_organizations:
    print(organization)

Number of unique organizations in Funding_Details_OND: 10,605
Number of unmatched organizations: 2

Unmatched organizations:
+CityxChange smart city project
National University of Defense Technology


In [8]:
# =============================================================================
# Manually assign classes to unmatched organizations
# =============================================================================

manual_class_mapping = {
    "+CityxChange smart city project": 0,
    "National University of Defense Technology": 0
}

# manual_class_mapping = {
#     "- Brazilian National Postdoctoral ProgramCoordenacao de Aperfeicoamento de Pessoal de Nivel Superior": 0,
#     "=Nippon Foundation Nereus Program": 0
# }


# Add the manually assigned classes to the main mapping.
org_class_mapping.update(
    manual_class_mapping
)

In [9]:
# =============================================================================
# Match each organization with its binary class
# =============================================================================

def create_org_class_pairs(value):
    """
    Convert organization names into (organization, class) pairs.
    """

    organizations = parse_organization_list(value)

    org_class_pairs = []

    for organization in organizations:

        if organization in org_class_mapping:

            org_class_pairs.append(
                (
                    organization,
                    int(
                        org_class_mapping[
                            organization
                        ]
                    )
                )
            )

        else:

            org_class_pairs.append(
                (
                    organization,
                    None
                )
            )

    return org_class_pairs


In [10]:
# =============================================================================
# Create the new column
# =============================================================================

df[NEW_COLUMN] = df[
    FUNDING_COLUMN
].apply(
    create_org_class_pairs
)

In [11]:
# =============================================================================
# Check for any remaining unmatched organizations
# =============================================================================

remaining_unmatched = set()

for org_class_pairs in df[NEW_COLUMN]:

    for organization, organization_class in org_class_pairs:

        if organization_class is None:
            remaining_unmatched.add(
                organization
            )


print(
    f"Remaining unmatched organizations: "
    f"{len(remaining_unmatched):,}"
)

if remaining_unmatched:

    print(
        sorted(
            remaining_unmatched
        )
    )

Remaining unmatched organizations: 0


In [12]:
# =============================================================================
# Save the completed dataset
# =============================================================================

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Output saved to: "
    f"{OUTPUT_FILE}"
)

print(
    f"Number of rows: "
    f"{len(df):,}"
)

Output saved to: LIS_data_cleaned_with_OND_Class.csv
Number of rows: 80,794
